<a href="https://colab.research.google.com/github/anushyadevi247/DSA-0302-NLP/blob/main/ASSESSEMT_coding_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PROBLEM 1 — TOKENIZATION WITH A REGULAR EXPRESSION**

In [1]:
import re

N = int(input())

pattern = r"[A-Za-z]+(?:['-][A-Za-z]+)*"

for i in range(N):
    text = input()
    tokens = re.findall(pattern, text)

    if len(tokens) == 0:
        print(0)
    else:
        print(len(tokens), *tokens)


4
The dog's tail wagged -- twice!
5 The dog's tail wagged twice
state-of-the-art NLP systems, circa 2026.
4 state-of-the-art NLP systems circa
12345 ... ???
0
Don't re-use rock--roll O'Neill's x
6 Don't re-use rock roll O'Neill's x


**PROBLEM 2 — SIMULATING A FINITE STATE AUTOMATON**

In [2]:
n, k = map(int, input().split())
alphabet = input().split()
start = int(input())

data = list(map(int, input().split()))
m = data[0]
accepting = set(data[1:])

t = int(input())

transitions = {}

for _ in range(t):
    p, c, r = input().split()
    p = int(p)
    r = int(r)

    if (p, c) not in transitions:
        transitions[(p, c)] = set()

    transitions[(p, c)].add(r)

Q = int(input())

for _ in range(Q):
    s = input()

    if s == "#":
        s = ""

    current = {start}

    for c in s:
        next_states = set()

        for state in current:
            next_states.update(transitions.get((state, c), set()))

        current = next_states

        if not current:
            break

    if current & accepting:
        print("ACCEPT")
    else:
        print("REJECT")


5 3
b a!
0
1 4
5
0 b 1
1 a 2
2 a 3
3 a 3
3 ! 4
5
baa!
ACCEPT
baaa!
ACCEPT
ba!
REJECT
b
REJECT
baa
REJECT


**PROBLEM 3 — INFLECTIONAL MORPHOLOGY GENERATOR**

In [3]:
import re

L = int(input())

lexicon = {}

for _ in range(L):
    stem, feature, form = input().split()
    lexicon[(stem, feature)] = form

Q = int(input())

vowels = set("aeiou")

def vowel_groups(word):
    return len(re.findall(r"[aeiou]+", word))

def doublable(word):
    if len(word) < 3:
        return False

    if word[-1] in "wxy":
        return False

    if word[-3] not in vowels:
        return False

    if word[-2] in vowels:
        return False

    if word[-1] in vowels:
        return False

    return vowel_groups(word) == 1

def inflect(stem, feature):
    if (stem, feature) in lexicon:
        return lexicon[(stem, feature)]

    if feature in ("N+PL", "V+3SG"):
        if stem.endswith(("s", "x", "z", "ch", "sh")):
            return stem + "es"

        if len(stem) >= 2 and stem[-1] == "o" and stem[-2] not in vowels:
            return stem + "es"

        if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
            return stem[:-1] + "ies"

        return stem + "s"

    elif feature == "V+PROG":
        if len(stem) >= 3 and stem.endswith("e") and stem[-2] not in vowels:
            return stem[:-1] + "ing"

        if doublable(stem):
            return stem + stem[-1] + "ing"

        return stem + "ing"

    elif feature in ("V+PAST", "ADJ+COMP", "ADJ+SUP"):
        if feature == "V+PAST":
            suffix = "ed"
        elif feature == "ADJ+COMP":
            suffix = "er"
        else:
            suffix = "est"

        if stem.endswith("e"):
            if suffix == "ed":
                return stem + "d"
            elif suffix == "er":
                return stem + "r"
            else:
                return stem + "st"

        if len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
            return stem[:-1] + "i" + suffix

        if doublable(stem):
            return stem + stem[-1] + suffix

        return stem + suffix

    return stem

for _ in range(Q):
    stem, feature = input().split()
    print(inflect(stem, feature))

3
goose N+PL geese
go V+PAST went
good ADJ+COMP better
8
cat N+PL
cats
church N+PL
churches
baby N+PL
babies
stop V+PROG
stoping
make V+PROG
making
carry V+PAST
carried
goose N+PL
geese
go V+PAST
went


**PROBLEM 4 — FINITE-STATE MORPHOLOGICAL PARSING**

In [4]:
import re

S = int(input())

lexicon = []

for _ in range(S):
    stem, pos = input().split()
    lexicon.append((stem, pos))

Q = int(input())

vowels = set("aeiou")

def vowel_groups(word):
    return len(re.findall(r"[aeiou]+", word))

def doublable(word):
    if len(word) < 3:
        return False
    if word[-1] in "wxy":
        return False
    if word[-3] in vowels:
        return False
    if word[-2] not in vowels:
        return False
    if word[-1] in vowels:
        return False
    return vowel_groups(word) == 1

def generate(stem, pos):
    forms = {}

    if pos == "N":
        features = [("PL", "N+PL")]
    elif pos == "V":
        features = [
            ("3SG", "V+3SG"),
            ("PROG", "V+PROG"),
            ("PAST", "V+PAST")
        ]
    else:
        features = [
            ("COMP", "ADJ+COMP"),
            ("SUP", "ADJ+SUP")
        ]

    for feature, rule in features:
        if rule in ("N+PL", "V+3SG"):
            if stem.endswith(("s", "x", "z", "ch", "sh")):
                form = stem + "es"
            elif len(stem) >= 2 and stem.endswith("o") and stem[-2] not in vowels:
                form = stem + "es"
            elif len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
                form = stem[:-1] + "ies"
            else:
                form = stem + "s"

        elif feature == "PROG":
            if len(stem) >= 3 and stem.endswith("e") and stem[-2] not in vowels:
                form = stem[:-1] + "ing"
            elif doublable(stem):
                form = stem + stem[-1] + "ing"
            else:
                form = stem + "ing"

        else:
            if stem.endswith("e"):
                if feature == "PAST":
                    form = stem + "d"
                elif feature == "COMP":
                    form = stem + "r"
                else:
                    form = stem + "st"

            elif len(stem) >= 2 and stem.endswith("y") and stem[-2] not in vowels:
                suffix = {
                    "PAST": "ed",
                    "COMP": "er",
                    "SUP": "est"
                }[feature]
                form = stem[:-1] + "i" + suffix

            elif doublable(stem):
                suffix = {
                    "PAST": "ed",
                    "COMP": "er",
                    "SUP": "est"
                }[feature]
                form = stem + stem[-1] + suffix

            else:
                suffix = {
                    "PAST": "ed",
                    "COMP": "er",
                    "SUP": "est"
                }[feature]
                form = stem + suffix

        forms[form] = feature

    return forms

analyses = {}

for stem, pos in lexicon:
    analyses.setdefault(stem, set()).add(stem + "+" + pos)

    generated = generate(stem, pos)

    for form, feature in generated.items():
        analysis = stem + "+" + pos + "+" + feature
        analyses.setdefault(form, set()).add(analysis)

for _ in range(Q):
    surface = input().strip()
    result = sorted(analyses.get(surface, set()))

    if result:
        print(", ".join(result))
    else:
        print("UNKNOWN")

5
cat N
watch N
watch V
study V
big ADJ
8
cats
cat+N+PL
watches
watch+N+PL, watch+V+3SG
watching
watch+V+PROG
studied
study+V+PAST
bigger
big+ADJ+COMP
cat
cat+N
biggest
big+ADJ+SUP
elephants
UNKNOWN


**PROBLEM 5 — THE PORTER STEMMER, STEP**

In [5]:
def is_consonant(word, i):
    ch = word[i]

    if ch in "aeiou":
        return False

    if ch == 'y':
        if i == 0:
            return True
        return not is_consonant(word, i - 1)

    return True


def measure(word):
    m = 0
    i = 0
    n = len(word)

    while i < n and is_consonant(word, i):
        i += 1

    while i < n:
        while i < n and not is_consonant(word, i):
            i += 1

        if i >= n:
            break

        m += 1

        while i < n and is_consonant(word, i):
            i += 1

    return m


def has_vowel(word):
    for i in range(len(word)):
        if not is_consonant(word, i):
            return True
    return False


def double_consonant(word):
    n = len(word)
    return (
        n >= 2
        and word[-1] == word[-2]
        and is_consonant(word, n - 1)
    )


def cvc(word):
    n = len(word)

    if n < 3:
        return False

    return (
        is_consonant(word, n - 3)
        and not is_consonant(word, n - 2)
        and is_consonant(word, n - 1)
        and word[-1] not in "wxy"
    )


def porter_step1(word):

    if word.endswith("sses"):
        word = word[:-2]
    elif word.endswith("ies"):
        word = word[:-2]
    elif word.endswith("ss"):
        pass
    elif word.endswith("s"):
        word = word[:-1]

    if word.endswith("eed"):
        stem = word[:-3]
        if measure(stem) > 0:
            word = stem + "ee"

    elif word.endswith("ed"):
        stem = word[:-2]
        if has_vowel(stem):
            word = stem
            word = cleanup(word)

    elif word.endswith("ing"):
        stem = word[:-3]
        if has_vowel(stem):
            word = stem
            word = cleanup(word)

    if word.endswith("y") and has_vowel(word[:-1]):
        word = word[:-1] + "i"

    return word


def cleanup(word):
    if word.endswith(("at", "bl", "iz")):
        return word + "e"

    if double_consonant(word) and word[-1] not in "lsz":
        return word[:-1]

    if measure(word) == 1 and cvc(word):
        return word + "e"

    return word


Q = int(input())

for _ in range(Q):
    word = input().strip()
    print(porter_step1(word))

8
caresses
caress
ponies
poni
cats
cat
feed
feed
agreed
agree
motoring
motor
hopping
hop
happy
happi
